# RQ4 — Showcase Evaluation & Calibration

**Research Question**: Does the meta-learner generalize to completely unseen datasets?

Evaluates the trained Option A meta-learner on 14 held-out showcase datasets that
were never part of meta-training: the original 9 plus 5 added to address the
review comment that an 8/9-dataset external evaluation is too small to support a
generalization claim. Downloads retry with backoff on transient network failures
(the original run silently dropped Credit Card Fraud after an `IncompleteRead`).

For each completed dataset, the notebook extracts Option A meta-features,
predicts the best method with the classifier, predicts expected LSE per method
with the regressor, runs all six methods where feasible, and compares predicted,
oracle, and k-means outcomes. A near-tie analysis (tolerance 0.02 LSE) separates
genuinely wrong recommendations from ties where the "wrong" method was equally
good in practice.

**Outputs**: `outputs/figures/showcase_comparison.csv` and `outputs/figures/lse_calibration.png`


In [1]:
import os, sys, pickle, warnings, time, shutil
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

ROOT        = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR     = os.path.join(ROOT, 'data', 'raw')
MODELS_DIR  = os.path.join(ROOT, 'outputs', 'models')
FIGURES_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(ROOT, 'src'))
openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)
print('Paths OK')

Paths OK


In [2]:
from lse import compute_lse, groundtruth_accuracy
from clustering import (
    pseudo_kmeans, pseudo_dbscan, pseudo_agglomerative,
    pseudo_gmm, pseudo_autoencoder, pseudo_dictlearn,
)
from metafeatures import extract_optA, extract_optC2
print('Modules loaded')

Modules loaded


In [3]:
SHOWCASE_DATASETS = [
    {'id': 61,    'name': 'Iris'},
    {'id': 187,   'name': 'Wine'},
    {'id': 15,    'name': 'Breast Cancer Wisconsin'},
    {'id': 53,    'name': 'Heart Disease (UCI)'},
    {'id': 40966, 'name': 'Palmer Penguins'},
    {'id': 37,    'name': 'Diabetes (Pima)'},
    {'id': 54,    'name': 'Vehicle Silhouettes'},
    {'id': 1590,  'name': 'Adult Income'},
    {'id': 1597,  'name': 'Credit Card Fraud'},
    # Added to broaden the held-out set beyond the original 9 datasets
    # (Reviewer 1 / editor: "the eight-dataset external/showcase evaluation is small").
    # Verified against OpenML qualities before adding: purely numeric, 0% missing,
    # and cross-checked by name AND (n_instances, n_classes) against
    # data/meta_table/dataset_manifest.csv to rule out train/showcase leakage via a
    # differently-numbered OpenML copy of the same underlying dataset (this is why
    # 'segment' id=36 was rejected — it duplicates training id=40984 in all but id).
    {'id': 59,    'name': 'Ionosphere'},
    {'id': 182,   'name': 'Statlog Satellite (satimage)'},
    {'id': 30,    'name': 'Page Blocks'},  # swapped in for Texture (11 classes > our 10-class max)
    {'id': 28,    'name': 'Optdigits'},
    {'id': 44,    'name': 'Spambase'},
]

METHODS = {
    'kmeans'   : pseudo_kmeans,
    'dbscan'   : pseudo_dbscan,
    'agg'      : pseudo_agglomerative,
    'gmm'      : pseudo_gmm,
    'autoenc'  : pseudo_autoencoder,
    'dictlearn': pseudo_dictlearn,
}
METHOD_NAMES = list(METHODS.keys())
print(f'{len(SHOWCASE_DATASETS)} showcase datasets')

14 showcase datasets


In [4]:
with open(os.path.join(MODELS_DIR, 'meta_clf_optA.pkl'), 'rb') as f:
    clf_bundle = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'meta_reg_optA.pkl'), 'rb') as f:
    reg_bundle = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'meta_clf_optC2.pkl'), 'rb') as f:
    clf_bundle_c2 = pickle.load(f)

meta_clf         = clf_bundle['pipeline']
clf_feat_cols    = clf_bundle['feature_cols']
meta_reg         = reg_bundle['pipeline']
reg_feat_cols    = reg_bundle['feature_cols']
lse_cols         = reg_bundle['lse_cols']
meta_clf_c2      = clf_bundle_c2['pipeline']
clf_feat_cols_c2 = clf_bundle_c2['feature_cols']

print(f'Classifier (Option A) : {len(clf_feat_cols)} features')
print(f'Regressor  (Option A) : {len(reg_feat_cols)} features')
print(f'Classifier (Option C2): {len(clf_feat_cols_c2)} features')
print(f'LSE cols   : {lse_cols}')

Classifier (Option A) : 20 features
Regressor  (Option A) : 20 features
Classifier (Option C2): 27 features
LSE cols   : ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']


In [5]:
# Imputer for showcase datasets that may have NaN
# (showcase datasets are not filtered for missing values the way meta-training is)
_imputer = SimpleImputer(strategy='median')


def load_and_split(dataset_id, max_retries=4, retry_delay=8):
    """
    Download + split a showcase dataset. OpenML downloads of larger datasets
    occasionally fail mid-transfer (IncompleteRead) rather than raising a clean
    404 — this is a transient network issue, not a data problem, so it is worth
    retrying with backoff instead of dropping the dataset. Any partially-written
    cache file from a failed attempt is removed so the retry re-downloads
    cleanly instead of parsing a truncated file.
    """
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            ds = openml.datasets.get_dataset(
                dataset_id, download_data=True,
                download_qualities=False,
                download_features_meta_data=False,
            )
            break
        except Exception as e:
            last_err = e
            print(f'    download attempt {attempt}/{max_retries} failed: {e}')
            cache_dir = os.path.join(RAW_DIR, 'org', 'openml', 'www', 'datasets', str(dataset_id))
            if os.path.isdir(cache_dir):
                shutil.rmtree(cache_dir, ignore_errors=True)
            if attempt < max_retries:
                time.sleep(retry_delay * attempt)
    else:
        raise last_err

    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    X = X.select_dtypes(include=[np.number]).astype(float)
    # Impute NaN in showcase datasets before any computation
    if X.isnull().any().any():
        X = pd.DataFrame(
            SimpleImputer(strategy='median').fit_transform(X),
            columns=X.columns
        )
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc, test_size=0.2,
        random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def scale(X_tr, X_te):
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [6]:
records = []

for ds_info in SHOWCASE_DATASETS:
    did  = ds_info['id']
    name = ds_info['name']
    print(f'\n── {name} (id={did}) ──')

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        if X_tr.shape[1] == 0:
            print('  SKIP: no numeric features')
            continue

        X_tr_sc, X_te_sc = scale(X_tr, X_te)
        n_cls = len(np.unique(y_tr))
        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)
        print(f'  balanced_gt={gt_acc:.3f}  n_cls={n_cls}  n_tr={len(X_tr)}')

        # Step 1: meta-features (Option A; label-free, computed on scaled train data)
        feats = extract_optA(X_tr_sc, n_cls)
        feat_vec_clf = np.array([[feats.get(c, np.nan) for c in clf_feat_cols]])
        feat_vec_reg = np.array([[feats.get(c, np.nan) for c in reg_feat_cols]])

        # Step 2: classify (Option A -- deployed model)
        predicted_method = meta_clf.predict(feat_vec_clf)[0]
        print(f'  Predicted best method (A): {predicted_method}')

        # Step 2b: C2 (dictionary-learning) meta-features + LogReg classifier,
        # for comparison. C2+LogReg has better minority-class recall in training
        # LOO-CV (autoenc 0.38 vs 0.31, dictlearn 0.40 vs 0.20, dbscan 0.80 vs 0.60
        # for RF+A) despite statistically indistinguishable overall accuracy
        # (McNemar p=1.000, 05_meta_learner.ipynb) -- worth testing on held-out data
        # since RF+A was picked by raw accuracy alone, a criterion blind to this.
        feats_c2 = extract_optC2(X_tr_sc)
        feat_vec_clf_c2 = np.array([[feats_c2.get(c, np.nan) for c in clf_feat_cols_c2]])
        predicted_method_c2 = meta_clf_c2.predict(feat_vec_clf_c2)[0]
        print(f'  Predicted best method (C2): {predicted_method_c2}')

        # Step 3: regress
        predicted_lse_vec = meta_reg.predict(feat_vec_reg)[0]
        predicted_lse = dict(zip(
            [c.replace('LSE_', '') for c in lse_cols],
            predicted_lse_vec,
        ))
        print(f'  Predicted LSE: ' +
              '  '.join(f'{m}={v:.3f}' for m, v in predicted_lse.items()))

        # Step 4: run all 6 methods
        true_lse = {}
        for mname, fn in METHODS.items():
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse_val, _ = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=mname, dataset_name=name, verbose=False,
                )
                true_lse[mname] = round(lse_val, 4)
            except Exception as e:
                print(f'    {mname} FAILED: {e}')
                true_lse[mname] = np.nan

        # Step 5: outcomes
        valid_methods = {m: v for m, v in true_lse.items() if not np.isnan(v)}
        if not valid_methods:
            print('  All methods failed — skipping')
            continue

        oracle_method = max(valid_methods, key=valid_methods.get)
        oracle_lse_actual = valid_methods[oracle_method]
        pred_lse_actual   = true_lse.get(predicted_method, np.nan)
        pred_lse_actual_c2 = true_lse.get(predicted_method_c2, np.nan)
        kmeans_lse        = true_lse.get('kmeans', np.nan)

        print(f'  Oracle  : {oracle_method} → LSE={oracle_lse_actual:.3f}')
        print(f'  Predicted A  ({predicted_method}): LSE={pred_lse_actual:.3f}')
        print(f'  Predicted C2 ({predicted_method_c2}): LSE={pred_lse_actual_c2:.3f}')
        print(f'  k-means baseline: LSE={kmeans_lse:.3f}')

        rec = {
            'dataset'            : name,
            'dataset_id'         : did,
            'n_classes'          : n_cls,
            'gt_acc'             : round(gt_acc, 3),
            'predicted_method'   : predicted_method,
            'predicted_lse_actual': round(pred_lse_actual, 3) if not np.isnan(pred_lse_actual) else np.nan,
            'predicted_method_c2'   : predicted_method_c2,
            'predicted_lse_actual_c2': round(pred_lse_actual_c2, 3) if not np.isnan(pred_lse_actual_c2) else np.nan,
            'oracle_method'      : oracle_method,
            'oracle_lse'         : round(oracle_lse_actual, 3),
            'kmeans_lse'         : round(kmeans_lse, 3) if not np.isnan(kmeans_lse) else np.nan,
            'hit'                : predicted_method == oracle_method,
            'hit_c2'             : predicted_method_c2 == oracle_method,
            'gap_vs_oracle'      : round(oracle_lse_actual - pred_lse_actual, 3) if not np.isnan(pred_lse_actual) else np.nan,
            'gap_vs_oracle_c2'   : round(oracle_lse_actual - pred_lse_actual_c2, 3) if not np.isnan(pred_lse_actual_c2) else np.nan,
            'gain_vs_kmeans'     : round(pred_lse_actual - kmeans_lse, 3) if not np.isnan(pred_lse_actual) else np.nan,
            'gain_vs_kmeans_c2'  : round(pred_lse_actual_c2 - kmeans_lse, 3) if not np.isnan(pred_lse_actual_c2) else np.nan,
        }
        for m, v in true_lse.items():
            rec[f'true_lse_{m}'] = v
        for m, v in predicted_lse.items():
            rec[f'pred_lse_{m}'] = round(v, 4)

        records.append(rec)

    except Exception as e:
        print(f'  FAILED: {e}')

print('\nDone.')


── Iris (id=61) ──


  balanced_gt=0.900  n_cls=3  n_tr=120


  Predicted best method (A): gmm


  Predicted best method (C2): gmm
  Predicted LSE: kmeans=0.664  dbscan=0.550  agg=0.648  gmm=0.697  autoenc=0.669  dictlearn=0.537


  Oracle  : gmm → LSE=1.037
  Predicted A  (gmm): LSE=1.037
  Predicted C2 (gmm): LSE=1.037
  k-means baseline: LSE=0.889

── Wine (id=187) ──
  balanced_gt=1.000  n_cls=3  n_tr=142
  Predicted best method (A): gmm


...

  Predicted best method (C2): gmm
  Predicted LSE: kmeans=0.573  dbscan=0.267  agg=0.564  gmm=0.565  autoenc=0.528  dictlearn=0.465


  Oracle  : agg → LSE=0.752
  Predicted A  (kmeans): LSE=0.714
  Predicted C2 (gmm): LSE=0.692
  k-means baseline: LSE=0.714

── Spambase (id=44) ──


  balanced_gt=0.937  n_cls=2  n_tr=3680
  Predicted best method (A): kmeans


  Predicted best method (C2): agg
  Predicted LSE: kmeans=0.771  dbscan=0.677  agg=0.745  gmm=0.748  autoenc=0.701  dictlearn=0.674


  Oracle  : autoenc → LSE=0.910
  Predicted A  (kmeans): LSE=0.532
  Predicted C2 (agg): LSE=0.531
  k-means baseline: LSE=0.532

Done.


In [7]:
results_df = pd.DataFrame(records)

SUMMARY_COLS = [
    'dataset', 'n_classes', 'gt_acc',
    'predicted_method', 'predicted_lse_actual',
    'predicted_method_c2', 'predicted_lse_actual_c2',
    'oracle_method', 'oracle_lse', 'kmeans_lse',
    'hit', 'hit_c2', 'gap_vs_oracle', 'gap_vs_oracle_c2',
    'gain_vs_kmeans', 'gain_vs_kmeans_c2',
]
summary = results_df[SUMMARY_COLS].copy()
print(summary.to_string(index=False))

n = len(summary)
print(f'\n=== Option A (deployed: RF+A) ===')
print(f'Top-1 accuracy        : {summary["hit"].mean():.1%}  ({summary["hit"].sum()}/{n})')
print(f'Mean gap vs oracle    : {summary["gap_vs_oracle"].mean():.3f}')
print(f'Mean gain vs k-means  : {summary["gain_vs_kmeans"].mean():.3f}')

print(f'\n=== Option C2 (candidate: LogReg on dictionary-learning features) ===')
print(f'Top-1 accuracy        : {summary["hit_c2"].mean():.1%}  ({summary["hit_c2"].sum()}/{n})')
print(f'Mean gap vs oracle    : {summary["gap_vs_oracle_c2"].mean():.3f}')
print(f'Mean gain vs k-means  : {summary["gain_vs_kmeans_c2"].mean():.3f}')

OUT_PATH = os.path.join(FIGURES_DIR, 'showcase_comparison.csv')
results_df.to_csv(OUT_PATH, index=False)
print(f'\nSaved -> {OUT_PATH}')

                     dataset  n_classes  gt_acc predicted_method  predicted_lse_actual predicted_method_c2  predicted_lse_actual_c2 oracle_method  oracle_lse  kmeans_lse   hit  hit_c2  gap_vs_oracle  gap_vs_oracle_c2  gain_vs_kmeans  gain_vs_kmeans_c2
                        Iris          3   0.900              gmm                 1.037                 gmm                    1.037           gmm       1.037       0.889  True    True          0.000             0.000           0.148              0.148
                        Wine          3   1.000              gmm                 1.000                 gmm                    1.000        kmeans       1.000       1.000 False   False          0.000             0.000           0.000              0.000
     Breast Cancer Wisconsin          2   0.952           kmeans                 0.989                 gmm                    0.947           agg       1.000       0.989 False   False          0.010             0.052           0.000            

## Near-Tie Analysis

Top-1 accuracy scores a "miss" whenever the predicted method isn't the exact
oracle method, even when the two are essentially interchangeable (e.g. Wine:
predicted gmm vs. oracle kmeans, both at LSE=1.000, `gap_vs_oracle=0.000`).
This counts, across all misses, how many are within a small tie tolerance
of the oracle rather than genuinely wrong recommendations — the practical
distinction between "wrong algorithm" and "algorithm choice didn't matter
here". The tolerance (0.02 LSE) matches the tie zone already used for the
Top-2+tie metric in `src/meta_learner.py`.


In [8]:
TIE_EPS = 0.02

misses = summary[~summary['hit']].copy()
misses['near_tie'] = misses['gap_vs_oracle'] <= TIE_EPS
summary['effective_hit'] = summary['hit'] | (summary['gap_vs_oracle'] <= TIE_EPS)

misses_c2 = summary[~summary['hit_c2']].copy()
misses_c2['near_tie'] = misses_c2['gap_vs_oracle_c2'] <= TIE_EPS
summary['effective_hit_c2'] = summary['hit_c2'] | (summary['gap_vs_oracle_c2'] <= TIE_EPS)

n_miss = len(misses)
n_near_tie = int(misses['near_tie'].sum())

print(f'=== Near-tie analysis (tolerance = {TIE_EPS} LSE) - Option A ===\n')
print(f'Top-1 misses            : {n_miss}/{n} ({n_miss/n:.1%})')
print(f'  of which near-ties    : {n_near_tie}/{n_miss if n_miss else 1} '
      f'({(n_near_tie/n_miss if n_miss else 0):.1%} of misses)')
print(f'  genuinely wrong picks : {n_miss - n_near_tie}/{n} '
      f'({(n_miss - n_near_tie)/n:.1%})')
print()
print(f'Top-1 accuracy (exact)      : {summary["hit"].mean():.1%}  ({summary["hit"].sum()}/{n})')
print(f'Effective Top-1 (with ties) : {summary["effective_hit"].mean():.1%}  '
      f'({summary["effective_hit"].sum()}/{n})')

if n_miss:
    print('\nMisses, sorted by gap to oracle:')
    print(misses[['dataset', 'predicted_method', 'oracle_method', 'gap_vs_oracle', 'near_tie']]
          .sort_values('gap_vs_oracle').to_string(index=False))

n_miss_c2 = len(misses_c2)
n_near_tie_c2 = int(misses_c2['near_tie'].sum())

print(f'\n=== Near-tie analysis (tolerance = {TIE_EPS} LSE) - Option C2 ===\n')
print(f'Top-1 misses            : {n_miss_c2}/{n} ({n_miss_c2/n:.1%})')
print(f'  of which near-ties    : {n_near_tie_c2}/{n_miss_c2 if n_miss_c2 else 1} '
      f'({(n_near_tie_c2/n_miss_c2 if n_miss_c2 else 0):.1%} of misses)')
print(f'  genuinely wrong picks : {n_miss_c2 - n_near_tie_c2}/{n} '
      f'({(n_miss_c2 - n_near_tie_c2)/n:.1%})')
print()
print(f'Top-1 accuracy (exact)      : {summary["hit_c2"].mean():.1%}  ({summary["hit_c2"].sum()}/{n})')
print(f'Effective Top-1 (with ties) : {summary["effective_hit_c2"].mean():.1%}  '
      f'({summary["effective_hit_c2"].sum()}/{n})')

if n_miss_c2:
    print('\nMisses, sorted by gap to oracle:')
    print(misses_c2[['dataset', 'predicted_method_c2', 'oracle_method', 'gap_vs_oracle_c2', 'near_tie']]
          .sort_values('gap_vs_oracle_c2').to_string(index=False))

print('\n=== Head-to-head: where the two models disagree on correctness ===')
both_hit    = (summary['hit'] & summary['hit_c2']).sum()
only_a_hit  = (summary['hit'] & ~summary['hit_c2']).sum()
only_c2_hit = (~summary['hit'] & summary['hit_c2']).sum()
neither_hit = (~summary['hit'] & ~summary['hit_c2']).sum()
print(f'  Both correct       : {both_hit}/{n}')
print(f'  Only A correct     : {only_a_hit}/{n}')
print(f'  Only C2 correct    : {only_c2_hit}/{n}')
print(f'  Neither correct    : {neither_hit}/{n}')
if only_a_hit or only_c2_hit:
    disagree = summary[summary['hit'] != summary['hit_c2']]
    print('\nDisagreements:')
    print(disagree[['dataset', 'predicted_method', 'predicted_method_c2', 'oracle_method', 'hit', 'hit_c2']]
          .to_string(index=False))

=== Near-tie analysis (tolerance = 0.02 LSE) - Option A ===

Top-1 misses            : 10/14 (71.4%)
  of which near-ties    : 3/10 (30.0% of misses)
  genuinely wrong picks : 7/14 (50.0%)

Top-1 accuracy (exact)      : 28.6%  (4/14)
Effective Top-1 (with ties) : 50.0%  (7/14)

Misses, sorted by gap to oracle:
                     dataset predicted_method oracle_method  gap_vs_oracle  near_tie
                        Wine              gmm        kmeans          0.000      True
Statlog Satellite (satimage)           kmeans       autoenc          0.003      True
     Breast Cancer Wisconsin           kmeans           agg          0.010      True
                   Optdigits           kmeans           agg          0.038     False
                  Ionosphere              agg        kmeans          0.045     False
                Adult Income              gmm        kmeans          0.119     False
             Palmer Penguins           kmeans           agg          0.139     False
        

## Calibration Analysis

For each method, plot predicted LSE from the meta-regressor against actual showcase LSE. In the current run, overall calibration MAE is **0.1289**. Per-method MAE is best for dictionary learning (0.0733) and worst for k-means (0.1649).

The empirical best-method overoptimism estimate is -0.0662 on average, with standard deviation 0.2089. The notebook suggests a confidence floor of **predicted LSE > 0.52** for replacing the previous hardcoded 0.60 value.


In [9]:
if len(results_df) < 3:
    print('Too few showcase results for calibration analysis.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    axes = axes.ravel()
    cal_errors = {}

    for ci, m in enumerate(['kmeans', 'dbscan', 'agg', 'gmm', 'autoenc', 'dictlearn']):
        true_col = f'true_lse_{m}'
        pred_col = f'pred_lse_{m}'
        if true_col not in results_df or pred_col not in results_df:
            continue

        valid = results_df[[true_col, pred_col]].dropna()
        if len(valid) == 0:
            continue

        true_vals = valid[true_col].values
        pred_vals = valid[pred_col].values
        mae = float(np.abs(true_vals - pred_vals).mean())
        cal_errors[m] = mae

        ax = axes[ci]
        ax.scatter(pred_vals, true_vals, s=60, alpha=0.7, color='steelblue',
                   edgecolors='white', linewidths=0.5)
        lim = [min(pred_vals.min(), true_vals.min()) - 0.05,
               max(pred_vals.max(), true_vals.max()) + 0.05]
        ax.plot(lim, lim, 'k--', lw=1, alpha=0.5, label='Perfect calibration')
        ax.set_xlabel('Predicted LSE')
        ax.set_ylabel('Actual LSE')
        ax.set_title(f'{m}  (MAE={mae:.3f})')
        ax.set_xlim(lim)
        ax.set_ylim(lim)

    plt.suptitle('LSE Calibration: Predicted vs Actual per Method (Showcase datasets)',
                 fontsize=12)
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, 'lse_calibration.png')
    fig.savefig(path, dpi=130)
    plt.close()
    print(f'Saved → {path}')

    print('\n=== Calibration MAE per method ===')
    for m, mae in sorted(cal_errors.items(), key=lambda x: x[1]):
        print(f'  {m:10s}  MAE={mae:.4f}')
    overall_cal_mae = float(np.mean(list(cal_errors.values())))
    print(f'  Overall: MAE={overall_cal_mae:.4f}')

    # Empirical confidence floor
    # If predicted LSE < floor, the prediction is unreliable
    pred_best_vals = results_df.apply(
        lambda row: row.get(f'pred_lse_{row["predicted_method"]}', np.nan), axis=1
    ).dropna()
    actual_best_vals = results_df['predicted_lse_actual'].dropna()

    overoptimistic = pred_best_vals - actual_best_vals
    print(f'\n=== Overoptimism in predicted best-method LSE ===')
    print(f'  Mean overestimate: {overoptimistic.mean():+.4f}')
    print(f'  Std: {overoptimistic.std():.4f}')
    print(f'  Suggested confidence floor: predicted LSE > {(actual_best_vals.mean() - actual_best_vals.std()):.2f}')
    print('  (replace the hardcoded 0.60 in CLAUDE.md with this empirical value)')

Saved → C:\MLResearch\outputs\figures\lse_calibration.png

=== Calibration MAE per method ===
  dictlearn   MAE=0.0865
  dbscan      MAE=0.1128
  agg         MAE=0.1225
  autoenc     MAE=0.1458
  gmm         MAE=0.1472
  kmeans      MAE=0.1663
  Overall: MAE=0.1302

=== Overoptimism in predicted best-method LSE ===
  Mean overestimate: -0.0144
  Std: 0.2132
  Suggested confidence floor: predicted LSE > 0.49
  (replace the hardcoded 0.60 in CLAUDE.md with this empirical value)


In [10]:
print('=' * 60)
print('RQ4 SHOWCASE EVALUATION SUMMARY')
print('=' * 60)
print(f'  Showcase datasets evaluated : {len(results_df)}')
if len(results_df) > 0:
    hits = results_df['hit'].sum()
    eff_hits = summary['effective_hit'].sum()
    hits_c2 = results_df['hit_c2'].sum()
    eff_hits_c2 = summary['effective_hit_c2'].sum()
    print(f'  --- Option A (deployed: RF+A) ---')
    print(f'  Top-1 accuracy              : {hits}/{len(results_df)} = {hits/len(results_df):.1%}')
    print(f'  Effective Top-1 (tie={TIE_EPS}) : {eff_hits}/{len(results_df)} = {eff_hits/len(results_df):.1%}')
    print(f'  Mean gap vs oracle          : {results_df["gap_vs_oracle"].mean():.3f} LSE')
    print(f'  Mean gain vs k-means        : {results_df["gain_vs_kmeans"].mean():.3f} LSE')
    print(f'  --- Option C2 (candidate: LogReg on dictionary-learning features) ---')
    print(f'  Top-1 accuracy              : {hits_c2}/{len(results_df)} = {hits_c2/len(results_df):.1%}')
    print(f'  Effective Top-1 (tie={TIE_EPS}) : {eff_hits_c2}/{len(results_df)} = {eff_hits_c2/len(results_df):.1%}')
    print(f'  Mean gap vs oracle          : {results_df["gap_vs_oracle_c2"].mean():.3f} LSE')
    print(f'  Mean gain vs k-means        : {results_df["gain_vs_kmeans_c2"].mean():.3f} LSE')
print()
print('Figures saved to outputs/figures/')
print('Phase complete.')

RQ4 SHOWCASE EVALUATION SUMMARY
  Showcase datasets evaluated : 14
  --- Option A (deployed: RF+A) ---
  Top-1 accuracy              : 4/14 = 28.6%
  Effective Top-1 (tie=0.02) : 7/14 = 50.0%
  Mean gap vs oracle          : 0.096 LSE
  Mean gain vs k-means        : -0.006 LSE
  --- Option C2 (candidate: LogReg on dictionary-learning features) ---
  Top-1 accuracy              : 4/14 = 28.6%
  Effective Top-1 (tie=0.02) : 5/14 = 35.7%
  Mean gap vs oracle          : 0.116 LSE
  Mean gain vs k-means        : -0.025 LSE

Figures saved to outputs/figures/
Phase complete.
